In [1]:
!pip install rasterio numpy pandas


[notice] A new release of pip available: 22.2.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import rasterio
import numpy as np
import os
import pandas as pd
from rasterio.warp import reproject, Resampling


main_dir = r"./tif_copy"


def resample_band(src, target_transform, target_width, target_height):
    data = src.read(1)
    dst = np.empty((target_height, target_width), dtype=data.dtype)

    reproject(
        source=data,
        destination=dst,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=target_transform,
        dst_crs=src.crs,
        resampling=Resampling.bilinear
    )
    return dst


def stats(arr):
    """Return mean, max, min, std, safely."""
    return [
        np.nanmean(arr),
        np.nanmax(arr),
        np.nanmin(arr),
        np.nanstd(arr)
    ]


results = []

for root, dirs, files in os.walk(main_dir):

    rel_path = os.path.relpath(root, main_dir)
    parts = rel_path.split(os.sep)

    if rel_path == ".":
        continue

    # ID folder (example: 1200)
    ID = parts[0]

    # Subfolder (example: 20180605T045701)
    if len(parts) < 2:
        continue
    row_name = parts[1]

    # Required bands
    required_patterns = {
        "B02": ".B2.tif",   # BLUE
        "B03": ".B3.tif",   # GREEN
        "B04": ".B4.tif",   # RED
        "B08": ".B8.tif",   # NIR
        "B11": ".B11.tif",  # SWIR
        "SCL": ".SCL.tif"    # Cloud mask
    }

    band_files = {}
    for band, pattern in required_patterns.items():
        matches = [f for f in files if pattern in f]
        if matches:
            band_files[band] = os.path.join(root, matches[0])

    if len(band_files) < 6:
        continue

    print("\nProcessing:", ID, row_name)

    # Open bands
    src_b2 = rasterio.open(band_files["B02"])
    src_b3 = rasterio.open(band_files["B03"])
    src_b4 = rasterio.open(band_files["B04"])
    src_b8 = rasterio.open(band_files["B08"])
    src_b11 = rasterio.open(band_files["B11"])
    src_scl = rasterio.open(band_files["SCL"])

    # Reference transform (use B2)
    ref_transform, ref_w, ref_h = (
        src_b2.transform, src_b2.width, src_b2.height
    )

    # Read & resample bands
    B2 = src_b2.read(1)
    B3 = resample_band(src_b3, ref_transform, ref_w, ref_h)
    B4 = resample_band(src_b4, ref_transform, ref_w, ref_h)
    B8 = resample_band(src_b8, ref_transform, ref_w, ref_h)
    B11 = resample_band(src_b11, ref_transform, ref_w, ref_h)
    SCL = resample_band(src_scl, ref_transform, ref_w, ref_h)

    # -------------------------------------------------
    # Convert to float to support nan masking
    # -------------------------------------------------
    B2 = B2.astype("float32")
    B3 = B3.astype("float32")
    B4 = B4.astype("float32")
    B8 = B8.astype("float32")
    B11 = B11.astype("float32")

    # -------------------------------------------------
    # CLOUD MASK (set cloud pixels to NaN)
    # -------------------------------------------------
    cloud_mask = (
        (SCL == 3)  |  # Cloud Shadow
        (SCL == 7)  |  # Unclassified
        (SCL == 8)  |  # Cloud Medium Probability
        (SCL == 9)  |  # Cloud High Probability
        (SCL == 10) |  # Thin Cirrus
        (SCL == 11)    # Snow/Ice
    )

    B2[cloud_mask] = np.nan
    B3[cloud_mask] = np.nan
    B4[cloud_mask] = np.nan
    B8[cloud_mask] = np.nan
    B11[cloud_mask] = np.nan

    # Aliases
    blue = B2
    green = B3
    red = B4
    nir = B8
    swir = B11

    # -------------------------------------------------
    # ALL INDICES (OLD + NEW)
    # -------------------------------------------------

    NDVI = (nir - red) / (nir + red + 1e-10)
    GNDVI = (nir - green) / (nir + green + 1e-10)
    SAVI = 1.5 * (nir - red) / (nir + red + 0.5)
    ARVI = (nir - (2*red - blue)) / (nir + (2*red - blue) + 1e-10)
    EVI = 2.5 * (nir - red) / (nir + 6*red - 7.5*blue + 1 + 1e-10)
    MSAVI = (2*nir + 1 - np.sqrt((2*nir + 1)**2 - 8*(nir - red))) / 2
    GCI = (nir / green) - 1
    SIPI = (nir - red) / (nir - blue + 1e-10)
    NDWI = (green - swir) / (green + swir + 1e-10)
    CI = (nir - swir) / (nir + swir + 1e-10)

    # NEW INDICES
    NDWI2 = (nir - swir) / (nir + swir + 1e-10)
    MSI = swir / (nir + 1e-10)
    GLA = (2*green - red - blue) / (2*green + red + blue + 1e-10)
    WI = (green - blue) / (red - green + 1e-10)
    NGRDI = (green - red) / (green + red + 1e-10)
    RGBVI = (green*green - red*blue) / (green*green + red*blue + 1e-10)
    VARI = (green - red) / (green + red - blue + 1e-10)
    ExR = 1.4 * red - green
    ExG = 2 * green - red - blue
    ExGR = ExG - ExR

    # -------------------------------------------------
    # Save stats
    # -------------------------------------------------
    results.append([
        ID, row_name,

        *stats(NDVI),
        *stats(GNDVI),
        *stats(SAVI),
        *stats(ARVI),
        *stats(EVI),
        *stats(MSAVI),
        *stats(GCI),
        *stats(SIPI),
        *stats(NDWI),
        *stats(CI),
        *stats(NDWI2),
        *stats(MSI),
        *stats(GLA),
        *stats(WI),
        *stats(NGRDI),
        *stats(RGBVI),
        *stats(VARI),
        *stats(ExR),
        *stats(ExG),
        *stats(ExGR)
    ])


# -------------------------------------------------
# CREATE DATAFRAME
# -------------------------------------------------
columns = ["ID", "Folder"]

index_names = [
    "NDVI", "GNDVI", "SAVI", "ARVI", "EVI", "MSAVI",
    "GCI", "SIPI", "NDWI", "CI",
    "NDWI2", "MSI", "GLA", "WI", "NGRDI",
    "RGBVI", "VARI", "ExR", "ExG", "ExGR"
]

stats_names = ["_mean", "_max", "_min", "_std"]

for idx in index_names:
    for s in stats_names:
        columns.append(idx + s)

df = pd.DataFrame(results, columns=columns)

df.to_csv("final_satellite_indices_with_cloudmask.csv", index=False)

print("\nDONE!")
print("Cloud mask applied")
print("All indices calculated")
print("CSV saved as final_satellite_indices_with_cloudmask.csv")


Processing: 0 20180605T045701_20180605T045701_T44PKT

Processing: 0 20180608T050651_20180608T050651_T44PKT


C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:141: RuntimeWarning: invalid value encountered in divide
  GCI = (nir / green) - 1


C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:141: RuntimeWarning: invalid value encountered in divide
  GCI = (nir / green) - 1
C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:141: RuntimeWarning: invalid value encountered in divide
  GCI = (nir / green) - 1



Processing: 0 20180610T045659_20180610T045659_T44PKT

Processing: 0 20180615T045701_20180615T045701_T44PKT


C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:141: RuntimeWarning: invalid value encountered in divide
  GCI = (nir / green) - 1
C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:141: RuntimeWarning: invalid value encountered in divide
  GCI = (nir / green) - 1


...

C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:141: RuntimeWarning: invalid value encountered in divide
  GCI = (nir / green) - 1
C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:30: RuntimeWarning: Mean of empty slice
  np.nanmean(arr),
C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:31: RuntimeWarning: All-NaN slice encountered
  np.nanmax(arr),
C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:32: RuntimeWarning: All-NaN slice encountered
  np.nanmin(arr),
c:\Users\aasth\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:141: RuntimeWarning: invalid value encountered in divide
  GCI = (nir / green) - 1
C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:30: RuntimeWarning: Mean of empty slice


Processing: 2 20190213T050921_20190213T052353_T44PKT

Processing: 2 20190215T045909_20190215T051352_T44PKT

Processing: 2 20190218T050849_20190218T051729_T44PKT


C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:141: RuntimeWarning: invalid value encountered in divide
  GCI = (nir / green) - 1



Processing: 2 20190220T045831_20190220T050947_T44PKT

Processing: 2 20190223T050811_20190223T051829_T44PKT

Processing: 2 20190225T045759_20190225T051421_T44PKT

DONE!
Cloud mask applied
All indices calculated
CSV saved as final_satellite_indices_with_cloudmask.csv


C:\Users\aasth\AppData\Local\Temp\ipykernel_21328\3044119477.py:141: RuntimeWarning: invalid value encountered in divide
  GCI = (nir / green) - 1
